In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.linalg as la
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import scipy.constants as c
from scipy.stats import ks_2samp, ttest_ind
from scipy.integrate import trapezoid

# Constants
L = 14.0  # Box size in nm
N = 20  # Grid points
KC = - (c.hbar**2) / (2 * c.m_e) # Kinetic Coeffecient
KC /= c.e

# 1D grid for x, y, and z axes (same range for all axes here)
x = y = z = np.linspace(-L/2, L/2, N)

# Cartesian grid
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Actual radial system for Woods Saxon Potential
r_obs = np.sqrt(X**2 + Y**2 + Z**2)
θ_min = np.arccos(np.clip(Z / r_obs, -1, 1)).flatten('F').min() # This one is used to find the minimum value of theta

# Grid setup for Spherical Potential
r = np.geomspace(r_obs.flatten('F').min(), 7, N)
θ = np.linspace(θ_min, np.pi - θ_min, N)
Φ = np.linspace(0, 2 * np.pi, N, endpoint=False)

R, Θ, φ = np.meshgrid(r, θ, Φ, indexing='ij')

# Define Grid Spacings (Uniformly spaced systems)
Δx = Δy = Δz = L / (N-1)    # Ends are not connected to each other (in nm)
ΔΦ = 2 * np.pi / N # Ends are connected
Δθ = np.pi / (N-1) # Ends are not connected

# Create identity matrices
I = sp.identity(N, format='csr')

# Potential Parameters
VL_DAW = 10  # Depth of left Double Asymmetrical Well
VR_DAW = 7.5  # Depth of right Double Asymmetrical Well
x_i= 2 # Horizontal Distance of Double Asymmetrical Well from origin
sigmaL_DAW = 1.5  # Width of left Double Asymmetrical Well
sigmaR_DAW = 2  # Width of right Double Asymmetrical Well

V0_SP = 3  # Height of Spherical Potential barrier
R0_SP = 7  # Radius of Spherical Potential
sigma_SP = 0.5  # Width of Spherical Potential transition region

# Define SP(r)
def SP(r_obs): # Scaling r_obs is useless since sigma_SP and R0_SP are in nm
    return V0_SP / (1 + np.exp((r_obs - R0_SP) / sigma_SP)) # Woods-Saxon potential

# Define DAW(X, Y, Z)
def DAW(X, Y, Z):
    rL2 = ((X + x_i)**2 + Y**2 + Z**2) / sigmaL_DAW**2
    rR2 = ((X - x_i)**2 + Y**2 + Z**2) / sigmaR_DAW**2
    V_total = VL_DAW * np.exp(-rL2) + VR_DAW * np.exp(-rR2)
    return V_total

# Redefine Kronecker Products
def TP(A, B, C):
    KP = sp.kron(sp.kron(A, B), C, format='csr')
    return KP

# Define Uniform Derivative Operator
def Du(ΔA='', mode='', type=''):
    diag = np.ones(N)
    off_diag = np.ones(N-1)
    D = sp.diags([off_diag, off_diag], [-1, 1], format='csr')
    D2 = sp.diags([off_diag, -2 * diag, off_diag], [-1, 0, 1], format='csr')

    if mode == 'azimuthal':
        upper = lower = np.zeros(N)
        for i in range(N):
            i_m = (i-1) % N
            i_p = (i+1) % N
            h1 = Φ[i] - Φ[i_m]
            h2 = Φ[i_p] - Φ[i]
            lower[i] = 2 / (h1 * (h1 + h2))
            upper[i] = 2 / (h2 * (h1 + h2))
        D2 = D2.tolil()
        D2[0, -1] = lower[0]
        D2[-1, 0] = upper[-1]
        return D2 / ΔΦ**2

    elif mode == 'polar':
        if type == 'First-Order':
            return D / Δθ
        elif type == 'Second-Order':
            return D2 / Δθ**2
        else:
            raise NotImplementedError(f"{type} for polar is not valid use 'First-Order' or 'Second-Order'")

    else:
        if ΔA == '':
            raise ValueError("ΔA must be provided for Cartesian coordinates")
        return D2 / (ΔA * 1e-9)**2

# Define Derivative Operator for radial system
def D_r(mode):


    lower = main = upper = np.zeros(N)

    for i in range(1, N-1):
        h1 = r[i] - r[i-1]
        h2 = r[i+1] - r[i]

        if mode == 'First-Order':
            lower[i] = -h2 / (h1 * (h1 + h2))
            main[i]  = (h2 - h1) / (h1 * h2)
            upper[i] = h1 / (h2 * (h1 + h2))

        elif mode == 'Second-Order':
            lower[i] = 2 / (h1 * (h1 + h2))
            main[i]  = -2 / (h1 * h2)
            upper[i] = 2 / (h2 * (h1 + h2))

        else:
            raise NotImplementedError(f"{mode} is not valid, use 'First-Order' or 'Second-Order'")

    if mode == 'First-Order':
        # Neumann (∂ψ/∂r = 0) at i=0 using forward difference
        h = r[1] - r[0]
        main[0]  = -1 / h
        upper[0] = 1 / h

        # Dirichlet (ψ = 0) at i=N-1 → ψ(N-1) = 0
        main[N-1] = 1.0
        lower[N-1] = 0
        upper[N-1] = 0

    elif mode == 'Second-Order':
        # Neumann (∂ψ/∂r = 0) at i=0 using forward 2nd order approximate
        h1 = r[1] - r[0]
        h2 = r[2] - r[1]
        main[0]  = -2 / (h1 * (h1 + h2))
        upper[0] = 2 / (h1 * h2)
        upper[1] = -2 / (h2 * (h1 + h2))

        # Dirichlet (ψ = 0) at i=N:-1
        main[N-1] = 1.0
        lower[N-1] = 0
        upper[N-1] = 0

    # Assemble sparse matrix
    diagonals = [lower[1:], main, upper[:-1]]
    offsets = [-1, 0, 1]
    Dn = sp.diags(diagonals, offsets, format='csr')

    return Dn

# Define Observables as Matrices
def CM(s):
    M = sp.diags(s, 0, format='csr')
    return M

# Define Kinetic term in H_SP
def KE_SP():
    Dn_r = D_r('First-Order')
    Dn2_r = D_r('Second-Order')
    Du2_Φ = Du(mode='azimuthal')
    Du_θ = Du(mode='polar', type='First-Order')
    Du2_θ = Du(mode='polar', type='Second-Order')

    O_r1 = 2 * CM(1 / (r * 1e-9))
    O_r2 = CM(1 / (r * 1e-9)**2)
    O_sin = CM(1 / np.sin(θ)**2)
    O_tan = CM(1 / np.tan(θ))

    H_r = KC * TP(Dn_r @ O_r1, I, I)
    H_r += KC * TP(Dn2_r, I, I)

    H_Φ = KC * TP(O_r2, O_sin, Du2_Φ)

    H_θ = KC * TP(O_r2, Du_θ @ O_tan, I)
    H_θ += KC * TP(O_r2, Du2_θ, I)

    return H_r + H_Φ + H_θ

# Define Kinetic term in H_DAW
def KE_DAW():
    Du2_x = Du(ΔA=Δx)
    Du2_y = Du(ΔA=Δy)
    Du2_z = Du(ΔA=Δz)

    H_x = KC * TP(Du2_x, I, I)
    H_y = KC * TP(I, Du2_y, I)
    H_z = KC * TP(I, I, Du2_z)

    return H_x + H_y + H_z


# Define Hamiltonians
H_v = CM(SP(r_obs).flatten('F'))  # Flatten with order='F' to match meshgrid(indexing='ij') ordering
H_rΦθ = KE_SP()

H_KE = KE_DAW()
H_PE = CM(DAW(X, Y, Z).flatten('F'))    # Flatten with order='F' to match meshgrid(indexing='ij') ordering

H_SP = H_rΦθ + H_v
H_DAW = H_KE + H_PE

# Solve for first 20 Eigenvalues
n = 20
E_SP, Ψ_1 = spla.eigsh(H_SP.tocsc(), k=n, sigma=0.0, which='LM')
E_DAW, Ψ_2 = spla.eigsh(H_DAW.tocsc(), k=n, sigma=0.0, which='LM')

def printE(E, label):
    for i, e in enumerate(E):
        E_str = f"{label}({i})"
        exponent = int(f"{e:.0e}".split("e")[1])
        coefficient = e / (10**exponent)
        superscripts = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")
        formatted_E = f"{coefficient:.4f} × 10{str(exponent).translate(superscripts)}"
        print(f"{E_str} = {formatted_E} eV")

printE(E_SP, "E_SP")
printE(E_DAW, "E_DAW")

# Store results in a dictionary
results = {
    "Spherical": {"eigvals": E_SP, "eigvecs": Ψ_1},
    "Double Asymmetrical": {"eigvals": E_DAW, "eigvecs": Ψ_2}
}

# Compute statistics
mean_SP, std_SP = np.mean(E_SP), np.std(E_SP)
mean_DAW, std_DAW = np.mean(E_DAW), np.std(E_DAW)
ks_stat, ks_p = ks_2samp(E_SP, E_DAW)  # KS test
t_stat, t_p = ttest_ind(E_SP, E_DAW, equal_var=False)  # T-test

def print_stat(label, value, unit=""):
    """Prints values in scientific notation unless zero."""
    if value == 0:
        print(f"{label} = 0 {unit}".strip())
    else:
        exponent = int(f"{value:.0e}".split("e")[1])  # Extract exponent
        coefficient = value / (10**exponent)  # Extract coefficient
        superscripts = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")  # Unicode superscripts
        formatted_value = f"{coefficient:.4f} × 10{str(exponent).translate(superscripts)}"
        print(f"{label} = {formatted_value} {unit}".strip())

# Print results
print_stat("Spherical Potential: Mean", mean_SP, "eV")
print_stat("Spherical Potential: Std Dev", std_SP, "eV")
print_stat("Double Asymmetrical Well: Mean", mean_DAW, "eV")
print_stat("Double Asymmetrical Well: Std Dev", std_DAW, "eV")
print_stat("KS Test Statistic", ks_stat)
print_stat("KS Test p-value", ks_p)
print_stat("T-Test Statistic", t_stat)
print_stat("T-Test p-value", t_p)

# Define Weights
def compute_weights(eigvals):
    unique_vals, inverse_indices, counts = np.unique(eigvals, return_inverse=True, return_counts=True)
    weights = 1.0 / counts[inverse_indices]  # 1 / degeneracy
    weights /= np.sum(weights)  # Normalize
    return weights

# Compute weights for each Eigenvalue

weights_SP = compute_weights(E_SP)
weights_DAW = compute_weights(E_DAW)

# Define rho
def compute_density_matrix(eigvecs, weights, return_diagonal=False):
    """
    Compute the density matrix ρ = Σ w_i |ψ_i⟩⟨ψ_i| or its diagonal.

    Parameters:
        eigvecs (np.ndarray): Eigenvectors (shape: M x k)
        weights (np.ndarray): Weights (length k)
        return_diagonal (bool): If True, return only the diagonal of ρ

    Returns:
        np.ndarray: Full density matrix (M x M)"""

    if return_diagonal:
        # Diagonal of density matrix only
        return np.sum(weights[i] * np.abs(eigvecs[:, i])**2 for i in range(len(weights)))

    # Full matrix version: ρ = E W E†
    W = np.diag(weights)
    rho = eigvecs @ W @ eigvecs.conj().T
    rho /= np.trace(rho)
    return rho

rho_SP = compute_density_matrix(Ψ_1, weights_SP, return_diagonal=False)
rho_DAW =  compute_density_matrix(Ψ_2, weights_DAW, return_diagonal=False)

# von Neumann Entropy
def von_neumann_entropy(rho):
    E = np.linalg.eigvalsh(rho)
    E = E[E > 1e-16]
    return -np.sum(E * np.log2(E))

# Linear Entropy
def linear_entropy(rho):
    purity = np.trace(rho @ rho)
    return 1 - purity

# Define Systems
# For Spherical Potential
S_SP = von_neumann_entropy(rho_SP)
S_lin_SP = linear_entropy(rho_SP)

# For Double Asymmetrical Well
S_DAW = von_neumann_entropy(rho_DAW)
S_lin_DAW = linear_entropy(rho_DAW)

# Print results
print(f"von Neumann Entropy of SP = {S_SP:.10e}")
print(f"Linear Entropy of SP = {S_lin_SP:.10e}")

print(f"von Neumann Entropy of DAW = {S_DAW:.10e}")
print(f"Linear Entropy of DAW = {S_lin_DAW:.10e}")

# Compute Quantum Coherence using l1-norm Coherence
def norm_coherence(rho):
    return np.sum(np.abs(rho)) - np.trace(np.abs(rho)) # Sum of off-diag elements

# Norm Coherence
NC_SP = norm_coherence(rho_SP)
NC_DAW = norm_coherence(rho_DAW)

# Print Results
print(f"Norm Coherence of SP = {NC_SP:.10e}")
print(f"Norm Coherence of DAW = {NC_DAW:.10e}")

E_SP(0) = -2.1400 × 10⁻³ eV
E_SP(1) = 1.2542 × 10⁻³ eV
E_SP(2) = 8.7542 × 10⁻³ eV
E_SP(3) = 1.1459 × 10⁻² eV
E_SP(4) = 1.2729 × 10⁻² eV
E_SP(5) = 1.7305 × 10⁻² eV
E_SP(6) = 2.1198 × 10⁻² eV
E_SP(7) = 2.3033 × 10⁻² eV
E_SP(8) = 2.3563 × 10⁻² eV
E_SP(9) = 2.4369 × 10⁻² eV
E_SP(10) = 2.6233 × 10⁻² eV
E_SP(11) = 2.7417 × 10⁻² eV
E_SP(12) = 2.9015 × 10⁻² eV
E_SP(13) = 3.1882 × 10⁻² eV
E_SP(14) = 3.3797 × 10⁻² eV
E_SP(15) = 3.4824 × 10⁻² eV
E_SP(16) = 3.5887 × 10⁻² eV
E_SP(17) = 3.6407 × 10⁻² eV
E_SP(18) = 3.7789 × 10⁻² eV
E_SP(19) = 3.8556 × 10⁻² eV
E_DAW(0) = 1.4711 × 10⁻² eV
E_DAW(1) = 1.5910 × 10⁻² eV
E_DAW(2) = 1.5910 × 10⁻² eV
E_DAW(3) = 1.7759 × 10⁻² eV
E_DAW(4) = 1.9895 × 10⁻² eV
E_DAW(5) = 2.0802 × 10⁻² eV
E_DAW(6) = 2.0802 × 10⁻² eV
E_DAW(7) = 2.2307 × 10⁻² eV
E_DAW(8) = 2.2870 × 10⁻² eV
E_DAW(9) = 2.5651 × 10⁻² eV
E_DAW(10) = 2.6115 × 10⁻² eV
E_DAW(11) = 2.6115 × 10⁻² eV
E_DAW(12) = 2.7860 × 10⁻² eV
E_DAW(13) = 2.7860 × 10⁻² eV
E_DAW(14) = 2.8795 × 10⁻² eV
E_DAW(15) = 2.9198 × 10⁻

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.linalg as la
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import scipy.constants as c
from scipy.stats import ks_2samp, ttest_ind
from scipy.integrate import trapezoid

# Constants
L = 14.0  # Box size in nm
N = 20  # Grid points
KC = - (c.hbar**2) / (2 * c.m_e) # Kinetic Coeffecient
KC /= c.e

# 1D grid for x, y, and z axes (same range for all axes here)
x = y = z = np.linspace(-L/2, L/2, N)

# Cartesian grid
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Actual radial system for Woods Saxon Potential
r_obs = np.sqrt(X**2 + Y**2 + Z**2)
θ_min = np.arccos(np.clip(Z / r_obs, -1, 1)).flatten('F').min() # This one is used to find the minimum value of theta

# Grid setup for Spherical Potential
r = np.geomspace(r_obs.flatten('F').min(), 7, N)
θ = np.linspace(θ_min, np.pi - θ_min, N)
Φ = np.linspace(0, 2 * np.pi, N, endpoint=False)

R, Θ, φ = np.meshgrid(r, θ, Φ, indexing='ij')

# Define Grid Spacings (Uniformly spaced systems)
Δx = Δy = Δz = L / (N-1)    # Ends are not connected to each other (in nm)
ΔΦ = 2 * np.pi / N # Ends are connected
Δθ = np.pi / (N-1) # Ends are not connected

# Create identity matrices
I = sp.identity(N, format='csr')

# Potential Parameters
VL_DAW = 10  # Depth of left Double Asymmetrical Well
VR_DAW = 7.5  # Depth of right Double Asymmetrical Well
x_i= 2 # Horizontal Distance of Double Asymmetrical Well from origin
sigmaL_DAW = 1.5  # Width of left Double Asymmetrical Well
sigmaR_DAW = 2  # Width of right Double Asymmetrical Well

V0_SP = 3  # Height of Spherical Potential barrier
R0_SP = 7  # Radius of Spherical Potential
sigma_SP = 0.5  # Width of Spherical Potential transition region

# Define SP(r)
def SP(r_obs):
    return V0_SP / (1 + np.exp((r_obs - R0_SP) / sigma_SP))

# Define DAW(X, Y, Z)
def DAW(X, Y, Z):
    rL2 = ((X + x_i)**2 + Y**2 + Z**2) / sigmaL_DAW**2
    rR2 = ((X - x_i)**2 + Y**2 + Z**2) / sigmaR_DAW**2
    V_total = VL_DAW * np.exp(-rL2) + VR_DAW * np.exp(-rR2)
    return V_total

# Redefine Kronecker Products
def TP(A, B, C):
    KP = sp.kron(sp.kron(A, B), C, format='csr')
    return KP

# Define Uniform Derivative Operator
def Du(ΔA='', mode='', type=''):
    diag = np.ones(N)
    off_diag = np.ones(N-1)
    D = sp.diags([off_diag, off_diag], [-1, 1], format='csr')
    D2 = sp.diags([off_diag, -2 * diag, off_diag], [-1, 0, 1], format='csr')

    if mode == 'azimuthal':
        upper = lower = np.zeros(N)
        for i in range(N):
            i_m = (i-1) % N
            i_p = (i+1) % N
            h1 = Φ[i] - Φ[i_m]
            h2 = Φ[i_p] - Φ[i]
            lower[i] = 2 / (h1 * (h1 + h2))
            upper[i] = 2 / (h2 * (h1 + h2))
        D2 = D2.tolil()
        D2[0, -1] = lower[0]
        D2[-1, 0] = upper[-1]
        return D2 / ΔΦ**2

    elif mode == 'polar':
        if type == 'First-Order':
            return D / Δθ
        elif type == 'Second-Order':
            return D2 / Δθ**2
        else:
            raise NotImplementedError(f"{type} for polar is not valid use 'First-Order' or 'Second-Order'")

    else:
        if ΔA == '':
            raise ValueError("ΔA must be provided for Cartesian coordinates")
        return D2 / (ΔA * 1e-9)**2

# Define Derivative Operator for radial system
def D_r(mode):
    M = len(r)

    lower = main = upper = np.zeros(M)

    for i in range(1, M-1):
        h1 = r[i] - r[i-1]
        h2 = r[i+1] - r[i]

        if mode == 'First-Order':
            lower[i] = -h2 / (h1 * (h1 + h2))
            main[i]  = (h2 - h1) / (h1 * h2)
            upper[i] = h1 / (h2 * (h1 + h2))

        elif mode == 'Second-Order':
            lower[i] = 2 / (h1 * (h1 + h2))
            main[i]  = -2 / (h1 * h2)
            upper[i] = 2 / (h2 * (h1 + h2))

        else:
            raise NotImplementedError(f"{mode} is not valid, use 'First-Order' or 'Second-Order'")

    if mode == 'First-Order':
        # Neumann (∂ψ/∂r = 0) at i=0 using forward difference
        h = r[1] - r[0]
        main[0]  = -1 / h
        upper[0] = 1 / h

        # Dirichlet (ψ = 0) at i=N-1 → ψ(N-1) = 0
        main[N-1] = 1.0
        lower[N-1] = 0
        upper[N-1] = 0

    elif mode == 'Second-Order':
        # Neumann (∂ψ/∂r = 0) at i=0 using forward 2nd order approximate
        h1 = r[1] - r[0]
        h2 = r[2] - r[1]
        main[0]  = -2 / (h1 * (h1 + h2))
        upper[0] = 2 / (h1 * h2)
        upper[1] = -2 / (h2 * (h1 + h2))

        # Dirichlet (ψ = 0) at i=N-1
        main[N-1] = 1.0
        lower[N-1] = 0
        upper[N-1] = 0

    # Assemble sparse matrix
    diagonals = [lower[1:], main, upper[:-1]]
    offsets = [-1, 0, 1]
    Dn = sp.diags(diagonals, offsets, format='csr')

    return Dn

# Define Observables as Matrices
def CM(s):
    M = sp.diags(s, 0, format='csr')
    return M

# Define Kinetic term in H_SP
def KE_SP():
    Dn_r = D_r('First-Order')
    Dn2_r = D_r('Second-Order')
    Du2_Φ = Du(mode='azimuthal')
    Du_θ = Du(mode='polar', type='First-Order')
    Du2_θ = Du(mode='polar', type='Second-Order')

    O_r1 = 2 * CM(1 / (r * 1e-9))
    O_r2 = CM(1 / (r * 1e-9)**2)
    O_sin = CM(1 / np.sin(θ)**2)
    O_tan = CM(1 / np.tan(θ))

    H_r = KC * TP(Dn_r @ O_r1, I, I)
    H_r += KC * TP(Dn2_r, I, I)

    H_Φ = KC * TP(O_r2, O_sin, Du2_Φ)

    H_θ = KC * TP(O_r2, Du_θ @ O_tan, I)
    H_θ += KC * TP(O_r2, Du2_θ, I)

    return H_r + H_Φ + H_θ

# Define Kinetic term in H_DAW
def KE_DAW():
    Du2_x = Du(ΔA=Δx)
    Du2_y = Du(ΔA=Δy)
    Du2_z = Du(ΔA=Δz)

    H_x = KC * TP(Du2_x, I, I)
    H_y = KC * TP(I, Du2_y, I)
    H_z = KC * TP(I, I, Du2_z)

    return H_x + H_y + H_z

# Define Hamiltonians
H_v = CM(SP(r_obs).flatten('F'))  # Flatten with order='F' to match meshgrid(indexing='ij') ordering
H_rΦθ = KE_SP()

H_KE = KE_DAW()
H_PE = CM(DAW(X, Y, Z).flatten('F'))    # Flatten with order='F' to match meshgrid(indexing='ij') ordering

H_SP = H_rΦθ + H_v
H_DAW = H_KE + H_PE

# Define Tunnelling Probability

def tunneling_probability(type, n_levels=20):
    α = -2 * np.sqrt((2 * c.m_e) / c.hbar**2)  # Constant
    T_list = []  # store tunneling probabilities for each energy level

    if type == 'SP':
        V = H_v * c.e  # eV to J
        E_vals = np.linalg.eigvalsh(H_SP.toarray())[:n_levels] * c.e
        V_arr = V.toarray().diagonal()
        r_f = r_obs.flatten('F') * 1e-9

        for E in E_vals:
            mask = V_arr > E
            integrand = np.sqrt(V_arr[mask] - E)
            r_masked = r_f[mask]
            exponent = α * trapezoid(integrand, r_masked)
            T = np.exp(exponent)
            T_list.append(T)

    elif type == 'DAW':
        E_vals = np.linalg.eigvalsh(H_KE.toarray())[:n_levels] * c.e

        # Split Left and Right Well
        rL2 = ((X + x_i)**2 + Y**2 + Z**2) / sigmaL_DAW**2
        rR2 = ((X - x_i)**2 + Y**2 + Z**2) / sigmaR_DAW**2
        V_l = VL_DAW * np.exp(-rL2)
        V_r = VR_DAW * np.exp(-rR2)

        # Define them as matrices
        V_L = CM(V_l.flatten('F'))
        V_R = CM(V_r.flatten('F'))

        V1 = V_L * c.e
        V2 = V_R * c.e
        V1_arr = V1.toarray().diagonal()
        V2_arr = V2.toarray().diagonal()

        r_l = np.sqrt((X + x_i)**2 + Y**2 + Z**2).flatten('F') * 1e-9
        r_r = np.sqrt((X - x_i)**2 + Y**2 + Z**2).flatten('F') * 1e-9

        for E in E_vals:
            # Left well
            mask1 = V1_arr > E
            integrand1 = np.sqrt(V1_arr[mask1] - E)
            r_masked1 = r_l[mask1]
            exponent1 = α * trapezoid(integrand1, r_masked1)

            # Right well
            mask2 = V2_arr > E
            integrand2 = np.sqrt(V2_arr[mask2] - E)
            r_masked2 = r_r[mask2]
            exponent2 = α * trapezoid(integrand2, r_masked2) * 1e-9  # meters

            T = np.exp(exponent1 + exponent2)
            T_list.append(T)

    else:
        raise NotImplementedError(f"{type} is not valid, use 'SP' or 'DAW'")

    return np.array(T_list)

# Compute the tunneling probability list
T_SP_list = tunneling_probability('SP')
T_DAW_list = tunneling_probability('DAW')

# Print Results
for i, (T_sp, T_daw) in enumerate(zip(T_SP_list, T_DAW_list)):
    print(f"I = {i:2d} | SP = {100*T_sp:.10f} % | DAW = {100*T_daw:.10f} %")

I =  0 | SP = 100.0000000000 % | DAW = 113.3988606678 %
I =  1 | SP = 100.0000000000 % | DAW = 114.8842809958 %
I =  2 | SP = 100.0000000000 % | DAW = 114.8842809958 %
I =  3 | SP = 100.0000000000 % | DAW = 114.8842809958 %
I =  4 | SP = 100.0000000000 % | DAW = 104.1088599254 %
I =  5 | SP = 100.0000000000 % | DAW = 104.1088599253 %
I =  6 | SP = 100.0000000000 % | DAW = 104.1088599253 %
I =  7 | SP = 100.0000000000 % | DAW = 112.9877854747 %
I =  8 | SP = 100.0000000000 % | DAW = 112.9877854747 %
I =  9 | SP = 100.0000000000 % | DAW = 112.9877854747 %
I = 10 | SP = 100.0000000000 % | DAW = 110.3567163331 %
I = 11 | SP = 100.0000000000 % | DAW = 76.1871102140 %
I = 12 | SP = 100.0000000000 % | DAW = 76.1871102140 %
I = 13 | SP = 100.0000000000 % | DAW = 76.1871102139 %
I = 14 | SP = 100.0000000000 % | DAW = 76.1871102139 %
I = 15 | SP = 100.0000000000 % | DAW = 76.1871102137 %
I = 16 | SP = 100.0000000000 % | DAW = 76.1871102136 %
I = 17 | SP = 100.0000000000 % | DAW = 105.3699497954 